# Kiểm tra mẫu URL và HTML theo host

Notebook này lấy tối đa 20 URL mỗi host từ corpus, lưu HTML gốc và tạo manifest để tra lại từng trang. Mặc định notebook sẽ tải URL mới; HTML đã lưu được dùng lại khi chạy lại.

Trong `artifacts/host_template_audit/` chỉ tạo `pages.jsonl`, `host_summary.csv` và `raw/`. Trạng thái `html_response` chỉ xác nhận nhận được HTML, không xác nhận đã trích đúng nội dung chính. Đây là khảo sát pilot, chưa phải lần crawl toàn bộ corpus. Để cài thư viện: `uv sync`.

In [1]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlsplit
import gzip
import hashlib
import json
import time

import duckdb
import httpx
import pandas as pd
from bs4 import BeautifulSoup
from IPython.display import display

HERE = Path.cwd()
ROOT = HERE.parent if HERE.name == "notebooks" else HERE
CORPUS = ROOT / "dataset/links_corpus.parquet"
OUTPUT = ROOT / "artifacts/host_template_audit"
RAW_DIR = OUTPUT / "raw"
MANIFEST = OUTPUT / "pages.jsonl"
HOST_SUMMARY = OUTPUT / "host_summary.csv"

PER_HOST = 20  # Có thể tăng tối đa 50.
SEED = 42
WORKERS = 8
HOST_DELAY = 1.0
TIMEOUT = 15.0
DOWNLOAD = True  # False để chỉ dùng HTML đã có; không tải URL mới.
FORCE_REFRESH = False  # True để tải lại cả URL đã có HTML.

if not CORPUS.exists():
    raise FileNotFoundError(CORPUS)
if not 1 <= PER_HOST <= 50:
    raise ValueError("PER_HOST phải nằm trong khoảng 1–50")
OUTPUT.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(exist_ok=True)

## 1. Chọn mẫu

Lấy mẫu luân phiên giữa các nhóm URL theo đoạn đường dẫn đầu tiên. Đây chỉ là cách phân bố mẫu; nhóm URL không khẳng định template HTML.

In [2]:
db = duckdb.connect()
db.execute("""
    CREATE TEMP TABLE urls AS
    SELECT id AS doc_id,
           url,
           lower(regexp_extract(url, '^https?://([^/?#]+)', 1)) AS host,
           regexp_extract(url, '^https?://[^/]+/([^/?#]*)', 1) AS first_path
    FROM read_parquet(?)
""", [str(CORPUS)])
if db.execute("SELECT count(*) FROM urls WHERE host = ''").fetchone()[0]:
    raise ValueError("Có URL không xác định được host")
db.execute("""
    CREATE TEMP TABLE urls_with_group AS
    SELECT *, CASE
        WHEN first_path = '' THEN '(root)'
        WHEN regexp_full_match(first_path, '[0-9]+') THEN '(numeric)'
        WHEN length(first_path) > 60 THEN '(long-slug)'
        ELSE first_path
    END AS stratum
    FROM urls
""")

samples = db.execute("""
    WITH ranked AS (
        SELECT *, count(*) OVER (PARTITION BY host, stratum) AS group_size,
               row_number() OVER (
                   PARTITION BY host, stratum ORDER BY hash(doc_id, ?), doc_id
               ) AS position
        FROM urls_with_group
    ), chosen AS (
        SELECT *, row_number() OVER (
            PARTITION BY host ORDER BY position, group_size DESC, stratum, doc_id
        ) AS sample_order
        FROM ranked WHERE position <= ?
    )
    SELECT doc_id, url, host, stratum, sample_order
    FROM chosen WHERE sample_order <= ? ORDER BY host, sample_order
""", [SEED, PER_HOST, PER_HOST]).df()
corpus_counts = db.execute("""
    SELECT host, count(*) AS corpus_urls
    FROM urls GROUP BY host
""").df()
db.close()

sample_counts = samples.groupby("host").size().rename("sample_urls")
host_plan = corpus_counts.set_index("host").join(sample_counts).reset_index()
print(f"{len(host_plan)} host · {len(samples):,} URL mẫu")
display(host_plan.sort_values("corpus_urls", ascending=False))

97 host · 1,820 URL mẫu


,host,corpus_urls,sample_urls
83,www.cnkang.com,963438,20
91,www.120ask.com,918479,20
71,www.familydoctor.com.cn,447453,20
9,ask.39.net,337375,20
20,zysjonline.com,243347,20
...,...,...,...
1,mega.vietnamplus.vn,2,2
64,yanglao.familydoctor.com.cn,2,2
53,fk.99.com.cn,1,1
63,ypk.familydoctor.com.cn,1,1


## 2. Tải mẫu và lưu HTML gốc

Mỗi trang được lưu một lần trong `raw/`. `pages.jsonl` ghi một dòng cho mỗi URL mẫu, gồm trạng thái, URL sau redirect, mã phản hồi và đường dẫn tới HTML. Nếu một host gặp ba lỗi truy cập liên tiếp, các URL chưa tải tiếp theo được đánh dấu `skipped_host`.

In [3]:
previous = {}
if MANIFEST.exists():
    for line in MANIFEST.read_text(encoding="utf-8").splitlines():
        if line.strip():
            row = json.loads(line)
            previous[(row["doc_id"], row["url"])] = row


def content_type(headers):
    return (headers.get("content-type") or headers.get("Content-Type") or "").lower()


def page_state(status, final_url, source_url, mime, body):
    if status is None:
        return "fetch_error"
    if status != 200:
        return "http_error"
    if not body:
        return "empty_response"
    if mime and "html" not in mime:
        return "non_html"

    text = body[:8000].decode("utf-8", errors="ignore").lower()
    challenge = (
        "denied by http_custom" in text
        or "document.cookie" in text and "location.reload" in text
        or "verify you are human" in text
        or "captcha" in text and len(body) < 10000
    )
    if challenge:
        return "access_challenge"

    source = urlsplit(source_url)
    final = urlsplit(final_url or source_url)
    if source.path not in ("", "/") and final.path in ("", "/") and final.netloc == source.netloc:
        return "redirected_home"
    return "html_response"


def fetch_host(host_samples):
    rows = []
    consecutive_errors = 0
    with httpx.Client(
        timeout=TIMEOUT,
        follow_redirects=True,
        headers={"User-Agent": "MedXIR/0.1 (host template survey)"},
    ) as client:
        for sample in host_samples:
            old = previous.get((int(sample["doc_id"]), sample["url"]), {})
            old_raw = OUTPUT / old["raw_path"] if old.get("raw_path") else None
            cached = old_raw is not None and old_raw.is_file() and not FORCE_REFRESH

            if not cached and not DOWNLOAD:
                rows.append(old.copy() if old else {**sample, "state": "not_fetched"})
                continue
            if not cached and consecutive_errors >= 3:
                rows.append({**sample, "state": "skipped_host"})
                continue

            status = old.get("http_status") if cached else None
            final_url = old.get("final_url") if cached else sample["url"]
            mime = old.get("content_type", "") if cached else ""
            fetched_at = old.get("fetched_at") if cached else None
            error = None
            try:
                if cached:
                    with gzip.open(old_raw, "rb") as file:
                        body = file.read()
                    raw_path = old["raw_path"]
                    digest = old.get("sha256") or hashlib.sha256(body).hexdigest()
                else:
                    time.sleep(HOST_DELAY)
                    response = client.get(sample["url"])
                    body = response.content
                    status = response.status_code
                    final_url = str(response.url)
                    mime = content_type(response.headers)
                    fetched_at = datetime.now(timezone.utc).isoformat()
                    digest = hashlib.sha256(body).hexdigest()
                    raw_path = f"raw/{sample['doc_id']}-{digest[:16]}.html.gz"
                    with gzip.open(OUTPUT / raw_path, "wb") as file:
                        file.write(body)
            except (httpx.HTTPError, OSError) as exc:
                error = f"{type(exc).__name__}: {exc}"
                state = "fetch_error"
                consecutive_errors += 1
                rows.append({**sample, "state": state, "error": error})
                continue

            state = page_state(status, final_url, sample["url"], mime, body)
            is_access_error = (
                state == "access_challenge"
                or state == "http_error"
                and (status in (401, 403, 407, 408, 425, 429) or status >= 500)
            )
            if is_access_error:
                consecutive_errors += 1
            else:
                consecutive_errors = 0

            rows.append({
                **sample,
                "state": state,
                "http_status": status,
                "final_url": final_url,
                "content_type": mime,
                "body_bytes": len(body),
                "sha256": digest,
                "raw_path": raw_path,
                "fetched_at": fetched_at,
                "reused_raw": cached,
            })
    return rows

## 3. Ghi manifest gọn

Manifest là chỉ mục cho HTML gốc. Bảng theo host là tổng hợp từ manifest nên có thể tạo lại, không cần giữ các bảng mẫu và chữ ký DOM riêng.

In [4]:
results = []
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    jobs = [
        pool.submit(fetch_host, group.sort_values("sample_order").to_dict("records"))
        for host, group in samples.groupby("host")
    ]
    for job in as_completed(jobs):
        results.extend(job.result())

sample_order = {
    (int(row.doc_id), row.url): i
    for i, row in enumerate(samples.itertuples(index=False))
}
results.sort(key=lambda row: sample_order[(int(row["doc_id"]), row["url"])])

with MANIFEST.open("w", encoding="utf-8") as file:
    for row in results:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")

used_raw = {row["raw_path"] for row in results if row.get("raw_path")}
for raw_file in RAW_DIR.glob("*.html.gz"):
    relative_path = str(raw_file.relative_to(OUTPUT))
    if relative_path not in used_raw:
        raw_file.unlink()

pages_df = pd.DataFrame(results)
state_counts = pd.crosstab(pages_df["host"], pages_df["state"])
summary = host_plan.set_index("host").join(state_counts, how="left").fillna(0)
summary["attempted_urls"] = summary.get("html_response", 0)
for state in ["http_error", "fetch_error", "access_challenge", "empty_response", "non_html", "redirected_home"]:
    if state in summary:
        summary["attempted_urls"] += summary[state]
summary = summary.reset_index()
summary.to_csv(HOST_SUMMARY, index=False)

print(f"{len(pages_df):,} URL · {len(pages_df.host.unique())} host")
display(summary.sort_values("corpus_urls", ascending=False))
print("Manifest:", MANIFEST)
print("HTML gốc:", RAW_DIR)

1,820 URL · 97 host


,host,corpus_urls,sample_urls,access_challenge,empty_response,fetch_error,html_response,http_error,non_html,redirected_home,skipped_host,attempted_urls
83,www.cnkang.com,963438,20,0,2,0,17,1,0,0,0,20
91,www.120ask.com,918479,20,0,0,0,20,0,0,0,0,20
71,www.familydoctor.com.cn,447453,20,0,0,0,0,3,0,0,17,3
9,ask.39.net,337375,20,0,0,0,20,0,0,0,0,20
20,zysjonline.com,243347,20,0,0,0,0,3,0,0,17,3
...,...,...,...,...,...,...,...,...,...,...,...,...
1,mega.vietnamplus.vn,2,2,0,0,0,2,0,0,0,0,2
64,yanglao.familydoctor.com.cn,2,2,0,0,0,0,2,0,0,0,2
53,fk.99.com.cn,1,1,0,0,0,1,0,0,0,0,1
63,ypk.familydoctor.com.cn,1,1,0,0,0,0,1,0,0,0,1


Manifest: /home/toska/code/personal/MedXIR/artifacts/host_template_audit/pages.jsonl
HTML gốc: /home/toska/code/personal/MedXIR/artifacts/host_template_audit/raw


## 4. Xem một trang mẫu

Đặt `DOC_ID` từ manifest rồi chạy cell bên dưới để xem text thô. Phần này chỉ xem HTML trong bộ nhớ, không tạo thêm file parse.

In [5]:
DOC_ID = None  # Ví dụ: 2300587
if DOC_ID is not None:
    row = pages_df.loc[pages_df.doc_id == DOC_ID].iloc[0].to_dict()
    print("URL:", row["url"])
    print("Trạng thái:", row["state"], "HTTP:", row.get("http_status"))
    if row.get("raw_path"):
        with gzip.open(OUTPUT / row["raw_path"], "rb") as file:
            body = file.read()
        soup = BeautifulSoup(body, "html.parser")
        for node in soup.select("script, style, noscript, template"):
            node.decompose()
        print("Tiêu đề:", soup.title.get_text(" ", strip=True) if soup.title else "")
        print(soup.get_text(" ", strip=True)[:3000])